In [5]:
%%writefile train_stylegan_attrib.py

# PASTE THE ENTIRE CODE HERE (all the code from the artifact)
#!/usr/bin/env python3
"""
Dual-branch EfficientNet-B0 for StyleGAN attribution (StyleGAN1/2/3)
Optimized for Kaggle with automatic train/val split and robust error handling

Usage in Kaggle notebook:
    !python train_stylegan_attrib.py --data_dir /kaggle/input/your-dataset/Fake --epochs 30
"""
import os
import gc
import argparse
from pathlib import Path
import numpy as np
from PIL import Image, ImageFile
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Allow loading of truncated/corrupted images
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import seaborn as sns


class StyleGANDataset(Dataset):
    """Dataset with FFT computation and robust error handling"""
    
    def __init__(self, samples, classes, size=224, augment=False):
        self.samples = samples
        self.classes = classes
        self.size = size
        self.augment = augment

        base_transforms = [transforms.Resize((size, size))]
        
        if augment:
            base_transforms.extend([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomResizedCrop(size, scale=(0.85, 1.0)),
                transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
            ])
        
        base_transforms.extend([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        
        self.rgb_transform = transforms.Compose(base_transforms)

    def __len__(self):
        return len(self.samples)

    def _compute_fft_mag(self, pil_image):
        """Compute FFT magnitude spectrum from grayscale image"""
        try:
            im_gray = pil_image.convert("L").resize((self.size, self.size), Image.BILINEAR)
            arr = np.asarray(im_gray, dtype=np.float32) / 255.0
            
            # 2D FFT with shift
            f = np.fft.fft2(arr)
            fshift = np.fft.fftshift(f)
            mag = np.log1p(np.abs(fshift))
            
            # Normalize to [0, 1]
            mag_min, mag_max = mag.min(), mag.max()
            if mag_max > mag_min:
                mag = (mag - mag_min) / (mag_max - mag_min)
            else:
                mag = np.zeros_like(mag)
            
            return mag.astype(np.float32)
        except Exception as e:
            print(f"FFT computation error: {e}")
            return np.zeros((self.size, self.size), dtype=np.float32)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        
        try:
            img = Image.open(path)
            if img.mode != 'RGB':
                img = img.convert("RGB")
        except Exception as e:
            print(f"Error loading {path}: {e}")
            # Create a dummy black image
            img = Image.new('RGB', (self.size, self.size), color=(0, 0, 0))
        
        # RGB branch
        try:
            rgb = self.rgb_transform(img)
        except Exception as e:
            print(f"Error in RGB transform: {e}")
            rgb = torch.zeros(3, self.size, self.size)
        
        # FFT branch
        try:
            mag = self._compute_fft_mag(img)
            mag_t = torch.from_numpy(mag).unsqueeze(0)  # [1, H, W]
            mag_t = mag_t.repeat(3, 1, 1)  # [3, H, W]
        except Exception as e:
            print(f"Error in FFT transform: {e}")
            mag_t = torch.zeros(3, self.size, self.size)
        
        return rgb, mag_t, label


def validate_image(img_path):
    """Check if image can be loaded and is valid"""
    try:
        with Image.open(img_path) as img:
            img.verify()
        # Reopen after verify (verify closes the file)
        with Image.open(img_path) as img:
            img.load()
            if img.size[0] < 10 or img.size[1] < 10:  # Skip tiny images
                return False
        return True
    except Exception:
        return False


def load_dataset_from_directory(root_dir, val_split=0.2, random_seed=42, validate_images=True):
    """
    Load dataset with automatic train/val split and optional image validation
    
    Args:
        root_dir: Path to directory containing class subdirectories
        val_split: Validation set fraction
        random_seed: Random seed for reproducibility
        validate_images: Whether to validate images before adding to dataset
    """
    root_path = Path(root_dir)
    
    if not root_path.exists():
        raise ValueError(f"Data directory does not exist: {root_dir}")
    
    # Find class directories
    all_dirs = [p for p in root_path.iterdir() if p.is_dir()]
    
    # Try to find StyleGAN directories
    class_dirs = [d for d in all_dirs if 'stylegan' in d.name.lower()]
    
    if not class_dirs:
        class_dirs = all_dirs
    
    if not class_dirs:
        raise ValueError(f"No class directories found in {root_dir}")
    
    classes = sorted([d.name for d in class_dirs])
    print(f"\n{'='*60}")
    print(f"Found {len(classes)} classes: {classes}")
    print(f"{'='*60}\n")
    
    # Collect samples
    all_samples = []
    image_extensions = {'.png', '.jpg', '.jpeg', '.bmp', '.PNG', '.JPG', '.JPEG', '.BMP'}
    
    for idx, class_name in enumerate(classes):
        class_path = root_path / class_name
        
        # Find all image files
        image_files = [f for f in class_path.iterdir() 
                      if f.is_file() and f.suffix in image_extensions]
        
        print(f"Class '{class_name}': {len(image_files)} images found")
        
        if validate_images and len(image_files) > 0:
            valid_count = 0
            invalid_count = 0
            
            for img_path in tqdm(image_files, desc=f"Validating {class_name}", leave=False):
                if validate_image(img_path):
                    all_samples.append((str(img_path), idx))
                    valid_count += 1
                else:
                    invalid_count += 1
            
            print(f"  ✓ Valid: {valid_count}, ✗ Invalid: {invalid_count}")
        else:
            for img_path in image_files:
                all_samples.append((str(img_path), idx))
    
    if len(all_samples) == 0:
        raise ValueError("No valid images found in dataset!")
    
    print(f"\n{'='*60}")
    print(f"Total valid samples: {len(all_samples)}")
    print(f"{'='*60}\n")
    
    # Stratified train/val split
    try:
        train_samples, val_samples = train_test_split(
            all_samples,
            test_size=val_split,
            random_state=random_seed,
            stratify=[s[1] for s in all_samples]
        )
    except ValueError as e:
        print(f"Warning: Stratified split failed ({e}), using random split")
        train_samples, val_samples = train_test_split(
            all_samples,
            test_size=val_split,
            random_state=random_seed
        )
    
    print(f"Train: {len(train_samples)} samples | Val: {len(val_samples)} samples\n")
    
    return train_samples, val_samples, classes


class DualEfficientNetB0(nn.Module):
    """Dual-branch architecture: RGB + FFT frequency domain"""
    
    def __init__(self, num_classes=3, pretrained=True, dropout=0.3):
        super().__init__()
        
        # RGB branch
        rgb_model = models.efficientnet_b0(pretrained=pretrained)
        self.rgb_backbone = nn.Sequential(
            rgb_model.features,
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
        
        # FFT branch (separate weights)
        fft_model = models.efficientnet_b0(pretrained=pretrained)
        self.fft_backbone = nn.Sequential(
            fft_model.features,
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
        
        # Fusion classifier
        feat_dim = 1280  # EfficientNet-B0 feature dimension
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim * 2, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, rgb, fft):
        rgb_feat = self.rgb_backbone(rgb)
        fft_feat = self.fft_backbone(fft)
        combined = torch.cat([rgb_feat, fft_feat], dim=1)
        out = self.classifier(combined)
        return out


def train_one_epoch(model, loader, criterion, optimizer, device, scaler=None):
    """Train for one epoch with mixed precision support"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc="Training", leave=False)
    
    for rgb, fft, labels in pbar:
        rgb = rgb.to(device, non_blocking=True)
        fft = fft.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        if scaler is not None:  # Mixed precision
            with torch.cuda.amp.autocast():
                logits = model(rgb, fft)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(rgb, fft)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
        
        running_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 
                         'acc': f'{correct/total:.4f}'})
    
    return running_loss / total, correct / total


def validate(model, loader, criterion, device):
    """Validate model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for rgb, fft, labels in tqdm(loader, desc="Validation", leave=False):
            rgb = rgb.to(device, non_blocking=True)
            fft = fft.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            logits = model(rgb, fft)
            loss = criterion(logits, labels)
            
            running_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    return running_loss / total, correct / total, np.array(all_preds), np.array(all_labels)


def plot_metrics(history, save_dir):
    """Plot training metrics"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax1.plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    ax2.plot(epochs, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
    ax2.plot(epochs, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, "training_metrics.png"), dpi=150, bbox_inches='tight')
    plt.close()


def plot_confusion(y_true, y_pred, classes, save_path):
    """Plot confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes,
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


def main(args):
    # Setup
    print("\n" + "="*60)
    print("StyleGAN Attribution Training")
    print("="*60 + "\n")
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    print(f"PyTorch version: {torch.__version__}")
    
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"CUDA version: {torch.version.cuda}")
        print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    
    # Load dataset
    print(f"\nLoading dataset from: {args.data_dir}")
    train_samples, val_samples, classes = load_dataset_from_directory(
        args.data_dir,
        val_split=args.val_split,
        random_seed=args.seed,
        validate_images=args.validate_images
    )
    
    # Create datasets
    train_ds = StyleGANDataset(train_samples, classes, size=args.size, augment=True)
    val_ds = StyleGANDataset(val_samples, classes, size=args.size, augment=False)
    
    # Create dataloaders
    train_loader = DataLoader(
        train_ds,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=True,
        drop_last=True
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=True
    )
    
    print(f"Batches per epoch: {len(train_loader)} train, {len(val_loader)} val")
    
    # Create model
    print("\nInitializing model...")
    model = DualEfficientNetB0(
        num_classes=len(classes),
        pretrained=True,
        dropout=args.dropout
    )
    model = model.to(device)
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=args.lr,
        weight_decay=args.weight_decay
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=3,
        verbose=True
    )
    
    # Mixed precision scaler
    scaler = torch.cuda.amp.GradScaler() if args.use_amp and torch.cuda.is_available() else None
    
    # Training loop
    os.makedirs(args.save_dir, exist_ok=True)
    best_val_loss = float('inf')
    best_val_acc = 0.0
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': []
    }
    
    print(f"\n{'='*60}")
    print(f"Starting training for {args.epochs} epochs")
    print(f"{'='*60}\n")
    
    for epoch in range(1, args.epochs + 1):
        print(f"\nEpoch {epoch}/{args.epochs}")
        print("-" * 40)
        
        # Train
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device, scaler
        )
        
        # Validate
        val_loss, val_acc, val_preds, val_labels = validate(
            model, val_loader, criterion, device
        )
        
        # Update scheduler
        scheduler.step(val_loss)
        
        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        # Print metrics
        print(f"Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}")
        print(f"Val   - Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            
            save_path = os.path.join(args.save_dir, "best_model.pth")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'classes': classes,
                'val_loss': val_loss,
                'val_acc': val_acc,
                'args': vars(args)
            }, save_path)
            print(f"✓ Saved best model (acc: {val_acc:.4f})")
        
        # Periodic reports
        if epoch % args.report_every == 0:
            print("\n" + "="*60)
            report = classification_report(
                val_labels, val_preds,
                target_names=classes,
                digits=4
            )
            print("Validation Classification Report:")
            print(report)
            print("="*60)
        
        # Clear cache
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    
    # Save final model
    final_path = os.path.join(args.save_dir, "final_model.pth")
    torch.save({
        'epoch': args.epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'classes': classes,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'args': vars(args)
    }, final_path)
    
    # Generate visualizations
    print("\n" + "="*60)
    print("Generating reports and visualizations...")
    print("="*60 + "\n")
    
    plot_metrics(history, args.save_dir)
    plot_confusion(val_labels, val_preds, classes, 
                   os.path.join(args.save_dir, "confusion_matrix.png"))
    
    # Save final report
    final_report = classification_report(
        val_labels, val_preds,
        target_names=classes,
        digits=4
    )
    
    report_path = os.path.join(args.save_dir, "classification_report.txt")
    with open(report_path, 'w') as f:
        f.write("="*60 + "\n")
        f.write("Final Classification Report\n")
        f.write("="*60 + "\n\n")
        f.write(final_report)
        f.write(f"\n\nBest Validation Accuracy: {best_val_acc:.4f}\n")
        f.write(f"Best Validation Loss: {best_val_loss:.4f}\n")
        f.write(f"\nTraining completed in {args.epochs} epochs\n")
    
    print("\n" + "="*60)
    print("✓ Training complete!")
    print("="*60)
    print(f"\nBest validation accuracy: {best_val_acc:.4f}")
    print(f"Results saved to: {args.save_dir}")
    print(f"  - best_model.pth")
    print(f"  - final_model.pth")
    print(f"  - training_metrics.png")
    print(f"  - confusion_matrix.png")
    print(f"  - classification_report.txt")
    print()


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Train StyleGAN attribution classifier",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter
    )
    
    # Paths
    parser.add_argument("--data_dir", type=str, 
                       default="/kaggle/input/stylegan123/Fake",
                       help="Root directory containing class folders")
    parser.add_argument("--save_dir", type=str, 
                       default="/kaggle/working",
                       help="Directory to save outputs")
    
    # Training
    parser.add_argument("--epochs", type=int, default=30,
                       help="Number of training epochs")
    parser.add_argument("--batch_size", type=int, default=32,
                       help="Batch size")
    parser.add_argument("--lr", type=float, default=1e-4,
                       help="Learning rate")
    parser.add_argument("--weight_decay", type=float, default=1e-4,
                       help="Weight decay")
    parser.add_argument("--dropout", type=float, default=0.3,
                       help="Dropout rate")
    
    # Data
    parser.add_argument("--size", type=int, default=224,
                       help="Image size")
    parser.add_argument("--val_split", type=float, default=0.2,
                       help="Validation split fraction")
    parser.add_argument("--seed", type=int, default=42,
                       help="Random seed")
    parser.add_argument("--validate_images", action="store_true", default=True,
                       help="Validate images before training")
    parser.add_argument("--no_validate_images", dest="validate_images", 
                       action="store_false",
                       help="Skip image validation (faster but risky)")
    
    # System
    parser.add_argument("--num_workers", type=int, default=2,
                       help="Number of data loading workers")
    parser.add_argument("--use_amp", action="store_true", default=True,
                       help="Use automatic mixed precision")
    parser.add_argument("--no_amp", dest="use_amp", action="store_false",
                       help="Disable mixed precision")
    
    # Logging
    parser.add_argument("--report_every", type=int, default=5,
                       help="Print classification report every N epochs")
    
    args = parser.parse_args()
    
    try:
        main(args)
    except Exception as e:
        print(f"\n{'='*60}")
        print(f"ERROR: {e}")
        print(f"{'='*60}\n")
        raise

Overwriting train_stylegan_attrib.py


In [6]:
!python train_stylegan_attrib.py --epochs 30 --batch_size 32


StyleGAN Attribution Training

Device: cuda
PyTorch version: 2.6.0+cu124
GPU: Tesla P100-PCIE-16GB
CUDA version: 12.4
Available GPU memory: 17.06 GB

Loading dataset from: /kaggle/input/stylegan123/Fake

Found 3 classes: ['StyleGAN', 'StyleGAN2', 'StyleGAN3']

Class 'StyleGAN': 7100 images found
  ✓ Valid: 7100, ✗ Invalid: 0                                                   
Class 'StyleGAN2': 7100 images found
  ✓ Valid: 7100, ✗ Invalid: 0                                                   
Class 'StyleGAN3': 7100 images found
  ✓ Valid: 7100, ✗ Invalid: 0                                                   

Total valid samples: 21300

Train: 17040 samples | Val: 4260 samples

Batches per epoch: 532 train, 134 val

Initializing model...
Total parameters: 9,327,867
Trainable parameters: 9,327,867

Starting training for 30 epochs


Epoch 1/30
----------------------------------------
Train - Loss: 0.2180, Acc: 0.9137                                               
Val   - Loss: 0.0266, Acc